# 08 - Ablation: straight-through clip
Clipped pixels get no gradient, and early on the OU guess is mostly clipped. Here the gradient passes the clip. Strengths tuned again. Run after `02`-`06`.

Needs `ou_guidance/` and `resnet_cifar10.pt` in `My Drive/tfg_ou/`. Runtime: T4 GPU.
If the install cell asks to restart, restart and run from the top.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/tfg_ou'      # code, checkpoint and results

In [ ]:
# pinned: a newer diffusers clashes with transformers
!pip install -q diffusers==0.35.2 transformers==4.55.2 "huggingface-hub>=0.34,<1.0" torch-fidelity

In [ ]:
import sys
sys.path.insert(0, f'{ROOT}/ou_guidance')
import helpers
helpers.ROOT = ROOT
from helpers import run, search, show_results, diagnose

In [ ]:
LABELS = [8]        # CIFAR10 classes (8 = ship)
N = 100             # images per final run
N_SEARCH = 64       # images per search trial (as in the TFG repo)
SWEEP_ST = [100, 30, 10]      # t*: guided last steps

In [ ]:
for label in LABELS:
    for family in ['simple', 'tfg']:                                             # OU-DPS, OU-TFG
        best = search('ou', family, label=label, n=N_SEARCH, straight_through=True)
        print(family, 'best settings:', best)
        for k in SWEEP_ST:
            run('ou', label=label, n=N, guided_steps=k, **best)

## Results

In [ ]:
show_results(kinds=['unconditional', 'DPS', 'TFG', 'OU-DPS', 'OU-DPS-ST', 'OU-TFG', 'OU-TFG-ST'])